## Load File

In [2]:
from langchain_community.document_loaders import PyPDFLoader



In [8]:
file_path= "../data/raw/AI Engineering.pdf"
loader= PyPDFLoader(file_path)
docs= loader.load()

docs[25]

Document(metadata={'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '../data/raw/AI Engineering.pdf', 'total_pages': 535, 'page': 25, 'page_label': '2'}, page_content='1 In this book, I use traditional ML to refer to all ML before foundation models.\nlarge-scale, readily available models brings about new possibilities and new chal‐\nlenges, which are the focus of this book.\nThis chapter begins with an overview of foundation models, the key catalyst behind\nthe explosion of AI engineering. I’ll then discuss a range of successful AI use cases,\neach illustrating what AI is good and not yet good at. As AI’s capabilities expand\ndaily, predicting its future possibilities becomes increasing

In [9]:
docs[25].page_content[:200]

'1 In this book, I use traditional ML to refer to all ML before foundation models.\nlarge-scale, readily available models brings about new possibilities and new chal‐\nlenges, which are the focus of this'

In [10]:
len(docs)

535

## Splitting document into chunks

- punkt_tab — A pre-trained NLTK model that knows how to split text into sentences. For example it knows that "Dr. Smith went to Washington. He loved it." is TWO sentences, not three (the period after "Dr" is not a sentence ending).

In [11]:
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
from langchain_core.documents import Document

MAX_CHARS = 1500

def truncate_to_limit(text, max_chars=MAX_CHARS):
    """Truncate text to max_chars but always end at a complete sentence."""
    if len(text) <= max_chars:
        return text
    
    # Cut to max_chars first
    truncated = text[:max_chars]
    
    # Find the last complete sentence within that limit
    sentences = sent_tokenize(truncated)
    
    # Drop the last sentence since it's likely cut off mid-way
    if len(sentences) > 1:
        sentences = sentences[:-1]
    
    return " ".join(sentences)


def split_into_paragraphs(docs, overlap=True):
    paragraphs = []
    for doc in docs:
        raw_paragraphs = [p.strip() for p in doc.page_content.split("\n\n") if p.strip()]

        for i, para in enumerate(raw_paragraphs):
            if overlap and i > 0:
                prev_para = raw_paragraphs[i - 1]
                sentences = sent_tokenize(prev_para)
                last_sentences = sentences[-2:] if len(sentences) >= 2 else sentences
                overlap_text = " ".join(last_sentences)
                content = overlap_text.strip() + " " + para
            else:
                content = para

            # Truncate cleanly at sentence boundary instead of mid-word/mid-sentence
            content = truncate_to_limit(content)

            paragraphs.append(
                Document(
                    page_content=content,
                    metadata=doc.metadata
                )
            )
    return paragraphs

splits = split_into_paragraphs(docs)

In [12]:
splits[0]

Document(metadata={'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '../data/raw/AI Engineering.pdf', 'total_pages': 535, 'page': 0, 'page_label': 'Cover'}, page_content='Chip Huyen\n AI Engineering\nBuilding Applications  \nwith Foundation Models')

In [13]:
len(splits)

529